# 11.07 - Image Transforms

**Notebook type:** Practice notebook with theory, exercises, TODO cells, and test cases.

**Daily output:** an augmentation grid showing 16 transformed examples, plus reusable `train_transforms` and `val_transforms` pipelines.

Today is about making image preprocessing reliable: resize, crop, flip, color jitter, tensor conversion, and normalization. The main habit is separating stochastic training transforms from deterministic validation transforms.


## Core Ideas

Transforms convert raw images into model-ready tensors while controlling what variation the model sees.

- **Resize:** makes the shortest or full image dimensions predictable before cropping.
- **Crop:** selects the spatial region. Random crop augments training data; center crop keeps validation stable.
- **Flip:** adds left-right invariance when the label should not change under reflection.
- **Color jitter:** varies brightness, contrast, saturation, or hue so the model is less tied to lighting.
- **ToTensor:** converts a PIL/NumPy image from `[H, W, C]` values in `0..255` to `[C, H, W]` float values in `0..1`.
- **Normalize:** subtracts channel means and divides by channel standard deviations.

A good validation transform should be deterministic. If validation metrics move just because transforms are random, the experiment becomes hard to trust.


In [ ]:
import csv
import os

import numpy as np
from PIL import Image, ImageDraw

import torch
import torchvision.transforms as T

SEED = 42
DATA_DIR = "_day11_image_data"
IMAGE_DIR = os.path.join(DATA_DIR, "images")
LABELS_CSV = os.path.join(DATA_DIR, "labels.csv")
MEAN = (0.485, 0.456, 0.406)
STD = (0.229, 0.224, 0.225)


def set_seed(seed=SEED):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)
print("torch:", torch.__version__, "torchvision transforms: available")


## Prepared Image Data

Run this cell before the exercises. It writes a deterministic toy image classification dataset into `_day11_image_data/` so the transform work can focus on preprocessing and visualization instead of data collection.


In [ ]:
def make_transform_demo_dataset(out_dir=DATA_DIR, image_size=48, n_per_class=4, seed=42):
    rng = np.random.default_rng(seed)
    image_dir = os.path.join(out_dir, "images")
    os.makedirs(image_dir, exist_ok=True)

    records = []
    classes = ["vertical", "horizontal", "diagonal"]
    base_colors = {
        "vertical": (220, 70, 80),
        "horizontal": (70, 170, 90),
        "diagonal": (65, 120, 220),
    }

    for label in classes:
        for idx in range(n_per_class):
            bg = np.zeros((image_size, image_size, 3), dtype=np.uint8)
            bg[:, :] = rng.integers(18, 55, size=3, dtype=np.uint8)
            img = Image.fromarray(bg)
            draw = ImageDraw.Draw(img)
            color = tuple(min(255, max(0, c + int(rng.integers(-20, 21)))) for c in base_colors[label])
            width = 5 + idx % 3

            if label == "vertical":
                x = int(image_size * (0.25 + 0.14 * idx))
                draw.rectangle([x, 5, x + width, image_size - 6], fill=color)
                draw.ellipse([image_size - 16, 8, image_size - 6, 18], fill=(245, 235, 90))
            elif label == "horizontal":
                y = int(image_size * (0.25 + 0.14 * idx))
                draw.rectangle([5, y, image_size - 6, y + width], fill=color)
                draw.ellipse([8, image_size - 18, 18, image_size - 8], fill=(245, 235, 90))
            else:
                offset = idx - 1
                draw.line([4, image_size - 8 - offset, image_size - 6, 5 + offset], fill=color, width=width)
                draw.rectangle([8, 8, 18, 18], outline=(245, 235, 90), width=2)

            noise = rng.normal(0, 7, size=(image_size, image_size, 3))
            arr = np.clip(np.asarray(img, dtype=np.float32) + noise, 0, 255).astype(np.uint8)
            img = Image.fromarray(arr)
            filename = f"{label}_{idx:02d}.png"
            rel_path = os.path.join("images", filename).replace(os.sep, "/")
            img.save(os.path.join(image_dir, filename))
            records.append({"image_path": rel_path, "label": label})

    with open(os.path.join(out_dir, "labels.csv"), "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["image_path", "label"])
        writer.writeheader()
        writer.writerows(records)

    return records


records = make_transform_demo_dataset()
print(f"Wrote {len(records)} images to {IMAGE_DIR}")


## Transform Utilities

These utilities support the exercises around approved `torchvision.transforms` pipelines. The notebook uses `torchvision` directly because it is listed in `RESOURCE/library.png` for image processing.


In [ ]:
def load_rgb_image(path):
    return Image.open(path).convert("RGB")


def apply_transform(transform, img, seed=None):
    if seed is not None:
        set_seed(seed)
    return transform(img)


def denormalize_tensor(tensor, mean=MEAN, std=STD):
    mean_t = torch.tensor(mean, dtype=tensor.dtype).view(3, 1, 1)
    std_t = torch.tensor(std, dtype=tensor.dtype).view(3, 1, 1)
    return tensor * std_t + mean_t


def tensor_to_uint8_image(tensor):
    tensor = tensor.detach().cpu().clamp(0, 1)
    arr = (tensor.permute(1, 2, 0).numpy() * 255).round().astype(np.uint8)
    return Image.fromarray(arr)


def stack_tensors(tensors):
    return torch.stack(list(tensors), dim=0)


## Exercise 11-A: Build Train and Validation Transforms

Use `torchvision.transforms.Compose` to build two pipelines. The training pipeline should include useful randomness such as crop, horizontal flip, and color jitter. The validation pipeline should be deterministic with resize, center crop, tensor conversion, and normalization.


In [ ]:
# TODO 11-A

def build_transform_pipelines(image_size=32, mean=MEAN, std=STD):
    raise NotImplementedError("Return torchvision train and validation transform pipelines.")


train_transforms, val_transforms = build_transform_pipelines()


## Exercise 11-B: Load Records and Transform a Batch

Read `labels.csv`, resolve each image path, load RGB images, apply a transform, and stack the transformed tensors into a batch.


In [ ]:
# TODO 11-B

def load_image_records(labels_csv=LABELS_CSV, data_dir=DATA_DIR):
    raise NotImplementedError("Read labels.csv and return records with full image paths and labels.")


def transform_batch(records, transform, n=4, seed=42):
    raise NotImplementedError("Load n images, apply transform, and return (batch_tensor, labels).")


## Exercise 11-C: Build the Augmentation Grid

Apply the training transform several times to the same source image, denormalize the tensors, convert them back to images, and save a 4-by-4 grid. The grid should make the training randomness visible.


In [ ]:
# TODO 11-C

def make_augmentation_grid(records, train_transform, index=0, n=16, cols=4, seed=42, out_path=os.path.join(DATA_DIR, "augmentation_grid.png")):
    raise NotImplementedError("Save and return a PIL grid image of n augmented samples.")


## Exercise 11-D: Compare Train vs Validation Behavior

Summarize repeated outputs from the train and validation transforms. Training should vary across seeds; validation should stay stable.


In [ ]:
# TODO 11-D

def summarize_transform_behavior(image_path, train_transform, val_transform, repeats=6, seed=42):
    raise NotImplementedError("Return a dictionary comparing train and validation transform variability.")


## Test Cases

Run this cell after completing the TODO cells above. A correct implementation should print `Day 11 tests passed`.


In [ ]:
def run_day11_tests():
    required_names = [
        "make_transform_demo_dataset",
        "load_rgb_image",
        "build_transform_pipelines",
        "load_image_records",
        "transform_batch",
        "make_augmentation_grid",
        "summarize_transform_behavior",
    ]
    for name in required_names:
        assert name in globals(), f"Missing function: {name}"
        assert callable(globals()[name]), f"{name} must be callable"

    assert os.path.exists(LABELS_CSV), "labels.csv was not created"
    assert os.path.isdir(IMAGE_DIR), "image directory was not created"
    image_files = sorted(name for name in os.listdir(IMAGE_DIR) if name.endswith(".png"))
    assert len(image_files) == 12, f"Expected 12 demo images, found {len(image_files)}"

    rows = load_image_records()
    assert len(rows) == 12, f"Expected 12 records, found {len(rows)}"
    labels = sorted({row["label"] for row in rows})
    assert labels == ["diagonal", "horizontal", "vertical"], labels
    assert all(os.path.exists(row["image_path"]) for row in rows), "Every record needs an existing image path"

    train_tf, val_tf = build_transform_pipelines(image_size=32)
    sample_img = load_rgb_image(rows[0]["image_path"])
    train_tensor = apply_transform(train_tf, sample_img.copy(), seed=123)
    val_tensor = apply_transform(val_tf, sample_img.copy(), seed=123)
    assert train_tensor.shape == (3, 32, 32), train_tensor.shape
    assert val_tensor.shape == (3, 32, 32), val_tensor.shape
    assert train_tensor.dtype == torch.float32
    assert val_tensor.dtype == torch.float32
    assert torch.isfinite(train_tensor).all()
    assert torch.isfinite(val_tensor).all()

    batch, batch_labels = transform_batch(rows, val_tf, n=5, seed=99)
    assert batch.shape == (5, 3, 32, 32), batch.shape
    assert len(batch_labels) == 5
    assert set(batch_labels).issubset({"vertical", "horizontal", "diagonal"})

    grid_path = os.path.join(DATA_DIR, "augmentation_grid.png")
    grid = make_augmentation_grid(rows, train_tf, index=0, n=16, cols=4, seed=7, out_path=grid_path)
    assert os.path.exists(grid_path), "augmentation grid was not saved"
    assert grid.size == (128, 128), grid.size

    summary = summarize_transform_behavior(rows[0]["image_path"], train_tf, val_tf, repeats=5, seed=500)
    assert summary["train_shape"] == (5, 3, 32, 32), summary["train_shape"]
    assert summary["val_shape"] == (5, 3, 32, 32), summary["val_shape"]
    assert summary["train_variability"] > 0.001, summary
    assert summary["val_variability"] < 1e-6, summary

    print("Day 11 tests passed")


run_day11_tests()


## Day 11 Checklist

Before trusting augmentation results, verify that training transforms include useful randomness, validation transforms are deterministic, tensors are `[C, H, W]`, normalized values are finite, labels do not change under augmentation, and the saved augmentation grid clearly shows 16 transformed examples.
